# T10 It7: does DINO ever drop the right place? A fixed coarse / DINO mixture at the same budget (3b)

Same fine-matching budget M = 10 per query: the pipeline (first 10 of the DINO order, T6 dumps) vs **the first 7 of the DINO order + the first
3 of the plain coarse order not already in** (`select_mixture(7, 3)`, runs `mix_official_try` = official TRY 185 photos, `mix_nudt101` = 100
NUDT 101 photos; the 6 standard queries). Per query: is a candidate within 4 sigma (by its coarse peak) among the matched places? Per photo:
the decision on the 6 standard queries (plain bar 8, and the NUDT-only judge at 0.9, no override), so the comparison is at the same queries.

In [1]:
import os, sys, glob
os.environ.setdefault("OMP_NUM_THREADS", "4")
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
import numpy as np, pandas as pd
pd.set_option("display.width", 250)
import train.checks.candidate_judge as cj
from train.checks.decision_signals import build_signal_table
from train.checks.rich_dump import load_rich
from train.checks.holdout import NUDT_DESIGN, load_t8_table
R = ROOT / "tests/Test_Iteration_10/results"
PAIRS = {"official_try": ("mix_official_try", "official"), "nudt101": ("mix_nudt101", "nudt_seed101")}
STD = {(round(h, 2), round(s, 2)) for h, s in cj.STANDARD}

## 1. Retrieval: is the right place among the 10 matched places of a query?

In [2]:
rows = []
for name, (mix, old) in PAIRS.items():
    for f in sorted(glob.glob(str(R / "rich_dump" / mix / "photo_*.npz"))):
        a = load_rich(f, with_matches=False); b = load_rich(ROOT / "tests/Test_Iteration_6/results/rich_dump" / old / os.path.basename(f), with_matches=False)
        bq = {(round(q["heading"], 2), round(q["scale"], 2)): q for q in b["queries"]}
        for q in a["queries"]:
            key = (round(q["heading"], 2), round(q["scale"], 2)); p = bq[key]
            err = lambda pk: np.hypot(pk[:, 1] - a["truth"][0], pk[:, 2] - a["truth"][1]) / a["sigma"]
            ea, eb = err(q["peaks"]), err(p["peaks"])
            rows.append(dict(data=name, photo=int(os.path.basename(f)[6:-4]), query=key,
                             dino10=bool((eb <= 4).any()), dino7=bool((eb[:7] <= 4).any()), dino5=bool((eb[:5] <= 4).any()),
                             mix=bool((ea <= 4).any()), mix_coarse3=bool((ea[7:] <= 4).any()),
                             dino10_right_inl=float(p["inliers"][eb <= 4].max()) if (eb <= 4).any() else 0.0,
                             mix_right_inl=float(q["inliers"][ea <= 4].max()) if (ea <= 4).any() else 0.0))
RET = pd.DataFrame(rows); RET.to_csv(R / "it7_retrieval.csv", index=False)
RET.groupby("data").agg(queries=("mix", "size"), dino5=("dino5", "mean"), dino7=("dino7", "mean"), dino10=("dino10", "mean"), mix_7_3=("mix", "mean"),
    dino_dropped_coarse_found=("mix_coarse3", lambda s: int((s & ~RET.loc[s.index, "dino10"]).sum())),
    mix_lost_dino8_10=("mix", lambda s: int((~s & RET.loc[s.index, "dino10"]).sum()))).round(4)

,queries,dino5,dino7,dino10,mix_7_3,dino_dropped_coarse_found,mix_lost_dino8_10
data,,,,,,,
nudt101,600,0.8417,0.8417,0.8483,0.8583,6,0
official_try,1110,0.7568,0.7568,0.7568,0.7568,0,0


## 2. Decision on the 6 standard queries: pipeline order vs mixture (same photos, same queries)

In [3]:
sig_dir = R / "signals"; sig_dir.mkdir(exist_ok=True)
D = load_t8_table(ROOT, locked=True)
p_all, judge = cj.judge_probability(D, NUDT_DESIGN, cj.FEATURES_CTX, "gbm")
rows = []
for name, (mix, old) in PAIRS.items():
    out = sig_dir / f"{mix}.csv.gz"
    if not out.exists():
        build_signal_table(R / "rich_dump" / mix, name, 4).to_csv(out, index=False)
    m = cj.add_context(cj.add_features(pd.read_csv(out)))
    ids = set(m.photo)
    o = D[(D.data == name) & D.photo.isin(ids)]
    o = cj.add_context(o[o.qkey.isin(STD)].drop(columns=["zgap", "ligap"]))      # pipeline order, the 6 standard queries, context from them only
    for label, t in [("pipeline: DINO first 10", o), ("mixture: DINO 7 + coarse 3", m)]:
        t = t.reset_index(drop=True)
        u = (t.inliers >= 3).to_numpy(); p = np.zeros(len(t)); p[u] = judge.predict_proba(t.loc[u, cj.FEATURES_CTX].to_numpy(float))[:, 1]
        for rule, r in [("bar 8", cj.baseline(t, 8, 99, min_choice=1)), ("judge 0.9, no override", cj.decide(t, p, 0.9, override=None, min_choice=3, wide_tau=1.01))]:
            s = cj.summary(r).loc[name]
            rows.append(dict(data=name, order=label, rule=rule, photos=int(s.n), score=s.score, wrong=int(s.wrong), answered=s.answered))
DEC = pd.DataFrame(rows); DEC.to_csv(R / "it7_decision_std6.csv", index=False); DEC

,data,order,rule,photos,score,wrong,answered
0,official_try,pipeline: DINO first 10,bar 8,185,50.400,0,0.859
1,official_try,pipeline: DINO first 10,"judge 0.9, no override",185,52.216,0,0.892
2,official_try,mixture: DINO 7 + coarse 3,bar 8,185,50.400,0,0.859
3,official_try,mixture: DINO 7 + coarse 3,"judge 0.9, no override",185,52.216,0,0.892
4,nudt101,pipeline: DINO first 10,bar 8,100,54.600,0,0.910
5,nudt101,pipeline: DINO first 10,"judge 0.9, no override",100,54.600,0,0.910
6,nudt101,mixture: DINO 7 + coarse 3,bar 8,100,55.200,0,0.920
7,nudt101,mixture: DINO 7 + coarse 3,"judge 0.9, no override",100,55.200,0,0.920
